In [ ]:
# ==============================================================================
# STEP 3: FINAL MERGE & APP PREPARATION
# 
# DESCRIPTION:
# This script takes the uncorrupted metadata from the master dataset and 
# merges it with the finalized 'category_human' decisions from the Excel audit.
# It strips out all internal CDC tracking columns to produce a clean, 
# app-ready Gold Layer dataset. It rebuilds the dataset entirely from scratch 
# to ensure all manual overrides are inherently captured.
# ==============================================================================

import os
import sys
import pandas as pd

print("\n" + "="*60)
print(" STEP 3: BUILDING FINAL APPLICATION DATASET ")
print("="*60)

# --- 1. SETUP & DIRECTORIES ---
notebook_dir = os.path.dirname(os.path.abspath(__file__)) if '__file__' in locals() else os.getcwd()
interim_dir = os.path.abspath(os.path.join(notebook_dir, "..", "..", "data", "interim"))
processed_dir = os.path.abspath(os.path.join(notebook_dir, "..", "..", "data", "processed"))

os.makedirs(processed_dir, exist_ok=True)

master_file = os.path.join(interim_dir, "master_ontology_dataset.csv")
human_review_file = os.path.join(interim_dir, "human_category_review.xlsx") 
final_app_file = os.path.join(processed_dir, "ontology_app_dataset.csv")

# --- 2. LOAD DATASETS ---
try:
    print("[*] Loading Master Metadata (Source of Truth)...")
    master_df = pd.read_csv(master_file, dtype={'CURIE': str})
    
    print(f"[*] Loading Human Reviewed Decisions from {os.path.basename(human_review_file)}...")
    review_df = pd.read_excel(human_review_file, dtype={'CURIE': str})
except FileNotFoundError as e:
    print(f"[!] CRITICAL ERROR: Required file missing. {e}")
    sys.exit(1)

# --- 3. MERGE LOGIC ---
print("[*] Merging human categories onto master metadata...")

if 'category_human' not in review_df.columns:
    print("[!] ERROR: 'category_human' column not found in the Excel file.")
    sys.exit(1)

# Left join maps the categories onto the pristine metadata
final_df = master_df.merge(review_df[['CURIE', 'category_human']], on='CURIE', how='left')

# --- 4. DATA LIFECYCLE FLAGS ---
def determine_status(row):
    val = str(row['category_human']).strip()
    # Check against string 'nan' and 'None' which Pandas can sometimes inject
    if val and val.lower() not in ['nan', 'none']:
        return val, "Human Reviewed"
    return "Uncategorized", "Pending Audit"

print("[*] Finalizing review status and cleaning schema...")
final_df[['working_category', 'review_status']] = final_df.apply(
    lambda r: pd.Series(determine_status(r)), axis=1
)

# --- 5. CLEANUP & EXPORT ---
# Drop the internal CDC column and the intermediate category column
cols_to_drop = ['row_hash', 'master_hash', 'category_human']
final_df = final_df.drop(columns=[c for c in cols_to_drop if c in final_df.columns])

print(f"[*] Exporting Gold Layer dataset to {os.path.basename(final_app_file)}...")
final_df.to_csv(final_app_file, index=False, encoding='utf-8-sig')

print("\n" + "="*60)
print(" DATASET COMPLETENESS SUMMARY ")
print("="*60)
status_counts = final_df['review_status'].value_counts()
for status, count in status_counts.items():
    print(f" - {status:<25}: {count:,} concepts")

print(f"\n[COMPLETE] Phase 3 Build Pipeline Finished.")
print("="*60)